In [2]:
# CELL 0 — Setup: dataset, đường dẫn, config, hàm đọc KFall
# =============================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks, optimizers
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

_BASES = ["/kaggle/input"]
try:
    import kagglehub
    _p = kagglehub.dataset_download("usmanabbasi2002/kfall-dataset")
    _BASES.insert(0, _p)
    print("kagglehub:", _p)
except Exception as e:
    print(f"[kagglehub bỏ qua: {type(e).__name__}]")


def find_named_subdir(name, bases=_BASES):
    for base in bases:
        if os.path.isdir(base):
            for cur, _d, _f in os.walk(base):
                if os.path.basename(cur).lower() == name.lower():
                    return cur
    return None


_sensor, _label = find_named_subdir("sensor_data"), find_named_subdir("label_data")
if _sensor is None:
    raise RuntimeError("Không dò được sensor_data. Gán tay DATA_ROOT / LABEL_ROOT.")
DATA_ROOT, LABEL_ROOT = _sensor, (_label or _sensor)
print("DATA_ROOT  =", DATA_ROOT)
print("LABEL_ROOT =", LABEL_ROOT)

# --- Config (khớp bài báo) ---------------------------------------------------
FS = 100
WIN_SIZE = 50                 # 0.5 s
STEP_TRAIN = 10               # 0.1 s — GIỮ NGUYÊN, đây là đòn bẩy pin 10x
FALL_LABEL_RATIO = 0.4
FEATURE_COLS = ["AccX", "AccY", "AccZ", "GyrX", "GyrY", "GyrZ"]
BATCH_SIZE = 64
CLIP_RANGE = 4.0
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# --- Đọc KFall ---------------------------------------------------------------
_FILE_RE = re.compile(r"S(\d+)T(\d+)R(\d+)\.csv$", re.IGNORECASE)
_label_cache = {}


def get_subject_ids():
    return sorted(d for d in os.listdir(DATA_ROOT)
                  if os.path.isdir(os.path.join(DATA_ROOT, d)))


def list_subject_trials(sid):
    d = os.path.join(DATA_ROOT, sid)
    out = []
    for f in sorted(os.listdir(d)):
        m = _FILE_RE.search(f)
        if m:
            out.append((int(m.group(2)), int(m.group(3)), os.path.join(d, f)))
    return out


def _to_int(v):
    if pd.isna(v):
        return None
    dg = "".join(c for c in str(v) if c.isdigit())
    return int(dg) if dg else None


def _parse_task_id(v):
    """'F01 (20)' -> 20. Số trong ngoặc mới là Task ID thật (khớp tên file S06T20R01)."""
    if pd.isna(v):
        return None
    m = re.search(r"\((\d+)\)", str(v))
    return int(m.group(1)) if m else _to_int(v)


def _pick_col(cols, keys):
    for c in cols:
        if any(k in str(c).lower() for k in keys):
            return c
    return None


def load_label_table(sid):
    if sid in _label_cache:
        return _label_cache[sid]
    path = os.path.join(LABEL_ROOT, f"{sid}_label.xlsx")
    if not os.path.isfile(path):
        _label_cache[sid] = None
        return None
    raw = pd.read_excel(path)
    cols = list(raw.columns)
    tc, rc = _pick_col(cols, ["task"]), _pick_col(cols, ["trial"])
    oc, ic = _pick_col(cols, ["onset"]), _pick_col(cols, ["impact"])
    if not all([tc, rc, oc, ic]):
        print(f"[Cảnh báo] thiếu cột nhãn {sid}: {cols}")
        _label_cache[sid] = None
        return None
    # Task Code bị merge-cell: chỉ dòng đầu mỗi khối 5 trial có giá trị.
    # Không ffill thì 4/5 trial ngã mất nhãn -> bị coi là ADL -> dự đoán đúng bị tính FP.
    raw[tc] = raw[tc].ffill()
    df = raw[[tc, rc, oc, ic]].copy()
    df.columns = ["task_id", "trial_id", "onset_frame", "impact_frame"]
    df["task_id"] = df["task_id"].apply(_parse_task_id)
    df["trial_id"] = df["trial_id"].apply(_to_int)
    df["onset_frame"] = pd.to_numeric(df["onset_frame"], errors="coerce")
    df["impact_frame"] = pd.to_numeric(df["impact_frame"], errors="coerce")
    df = df.dropna()
    for c in df.columns:
        df[c] = df[c].astype(int)
    _label_cache[sid] = df
    return df


def load_kfall_trial(sid, task, trial, path):
    df = pd.read_csv(path)
    miss = [c for c in FEATURE_COLS if c not in df.columns]
    if miss:
        raise KeyError(f"{path} thiếu {miss}. Cột thật: {list(df.columns)}")
    data = df[FEATURE_COLS].values.astype(np.float32)
    onset = impact = None
    lab = load_label_table(sid)
    if lab is not None:
        m = lab[(lab.task_id == task) & (lab.trial_id == trial)]
        if len(m):
            onset, impact = int(m.onset_frame.iloc[0]), int(m.impact_frame.iloc[0])
    return data, onset, impact


def make_windows(data, onset=None, impact=None, win=WIN_SIZE, step=STEP_TRAIN,
                 ratio=FALL_LABEL_RATIO):
    W, L = [], []
    for s in range(0, data.shape[0] - win + 1, step):
        e = s + win
        lab = 0
        if onset is not None and impact is not None:
            if max(0, min(e, impact) - max(s, onset)) / win >= ratio:
                lab = 1
        W.append(data[s:e]); L.append(lab)
    return W, L


_subs = get_subject_ids()
print(f"Subject: {len(_subs)} -> {_subs[:4]} ...")
assert _subs, "Không thấy subject nào."

TensorFlow: 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
kagglehub: /kaggle/input/datasets/usmanabbasi2002/kfall-dataset
DATA_ROOT  = /kaggle/input/datasets/usmanabbasi2002/kfall-dataset/KFall Dataset/KFall Dataset/sensor_data
LABEL_ROOT = /kaggle/input/datasets/usmanabbasi2002/kfall-dataset/KFall Dataset/KFall Dataset/label_data
Subject: 32 -> ['SA06', 'SA07', 'SA08', 'SA09'] ...


In [3]:
# CELL 1 — Dataset, chia subject, hằng số chuẩn hoá (GIỮ DỮ LIỆU THÔ cho augment)
# =============================================================================
SEED_SPLIT = 42
SEEDS = [0, 1, 2]

# Bạn đã chọn: seed chỉ đổi khởi tạo, split cố định.
# Số liệu v1 cho thấy chênh lệch val-test là 10 điểm specificity, tức phương sai
# subject lớn gấp ~4 lần phương sai khởi tạo. Đổi cờ này thành True nếu muốn std
# phản ánh khả năng khái quát sang người mới thay vì chỉ nhiễu khởi tạo.
VARY_SPLIT = False


def collect_fall_task_ids(subs):
    """Task ngã = union task_id trong các file nhãn (file nhãn chỉ có dòng cho trial ngã)."""
    ids = set()
    for s in subs:
        df = load_label_table(s)
        if df is not None and len(df):
            ids.update(df.task_id.unique().tolist())
    return ids


FALL_TASK_IDS = collect_fall_task_ids(_subs)
print(f"Task ngã ({len(FALL_TASK_IDS)}): {sorted(FALL_TASK_IDS)}")
assert FALL_TASK_IDS, "Không đọc được nhãn nào."


def build_dataset(subs, step=STEP_TRAIN):
    """Trả X thô (CHƯA chuẩn hoá), y cửa sổ, file_ids, impact_frames, file_true.

    file_true lấy từ TASK CODE, không suy từ nhãn cửa sổ — trial ngã có đoạn
    onset->impact quá ngắn sẽ không có cửa sổ nào đạt luật 40%, và bản cũ đẩy
    nhầm nó sang mẫu số ADL, hỏng cả sensitivity lẫn specificity.
    """
    X, y, fid = [], [], []
    imp, ftrue = {}, {}
    n_nopos = 0
    for s in subs:
        for task, trial, path in list_subject_trials(s):
            try:
                d, on, ip = load_kfall_trial(s, task, trial, path)
            except Exception as e:
                print(f"[Bỏ qua] {path}: {e}")
                continue
            is_fall = int(task in FALL_TASK_IDS)
            W, L = make_windows(d, on, ip, step=step)
            f = f"{s}_T{task:02d}_R{trial:02d}"
            X.extend(W); y.extend(L); fid.extend([f] * len(W))
            ftrue[f] = is_fall
            if is_fall:
                if ip is not None:
                    imp[f] = ip
                if max(L) == 0:
                    n_nopos += 1
    print(f"  file={len(ftrue)} (ngã={sum(ftrue.values())}, "
          f"ADL={len(ftrue)-sum(ftrue.values())})  cửa_sổ={len(X)}")
    if n_nopos:
        print(f"  [!] {n_nopos} trial ngã không có cửa sổ đạt luật 40%")
    return (np.array(X, np.float32), np.array(y, np.int64), np.array(fid), imp, ftrue)


def make_split(seed):
    tr, te = train_test_split(_subs, test_size=6, random_state=seed)
    trf, va = train_test_split(tr, test_size=0.20, random_state=seed)
    return trf, va, te


train_subs, val_subs, test_subs = make_split(SEED_SPLIT)
print(f"\nTrain-fit({len(train_subs)}) Val({len(val_subs)}) Test({len(test_subs)})")
print(f"Test = {test_subs}")

print("\n== Train =="); Xtr_raw, ytr, idtr, imptr, ftr = build_dataset(train_subs)
print("== Val ==");   Xva_raw, yva, idva, impva, fva = build_dataset(val_subs)
print("== Test ==");  Xte_raw, yte, idte, impte, fte = build_dataset(test_subs)

_flat = Xtr_raw.reshape(-1, 6)
GLOBAL_MEAN = _flat.mean(0).astype(np.float32)
GLOBAL_STD = (_flat.std(0) + 1e-8).astype(np.float32)
del _flat; gc.collect()


def normalize(X):
    return np.clip((X - GLOBAL_MEAN) / GLOBAL_STD, -CLIP_RANGE, CLIP_RANGE).astype(np.float32)


# Val/test chuẩn hoá một lần (dùng cho fit-validation, predict, TFLite).
# Train giữ thô để augment đúng vật lý — xoay trục phải làm TRƯỚC khi z-score,
# vì mỗi trục có thang đo khác nhau, xoay dữ liệu đã chuẩn hoá là vô nghĩa.
Xva, Xte = normalize(Xva_raw), normalize(Xte_raw)
del Xva_raw, Xte_raw; gc.collect()

ytr_oh = tf.keras.utils.to_categorical(ytr, 2)
yva_oh = tf.keras.utils.to_categorical(yva, 2)

print(f"\nGLOBAL_MEAN = {GLOBAL_MEAN.round(4).tolist()}")
print(f"GLOBAL_STD  = {GLOBAL_STD.round(4).tolist()}")
print(f"Tỉ lệ fall — train {ytr.mean():.4f} | val {yva.mean():.4f} | test {yte.mean():.4f}")
print("\n[Firmware] ESP32 PHẢI áp đúng GLOBAL_MEAN/STD và clip ±4 trên dữ liệu thô.")

Task ngã (15): [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]

Train-fit(20) Val(6) Test(6)
Test = ['SA36', 'SA21', 'SA30', 'SA23', 'SA14', 'SA15']

== Train ==
  file=3155 (ngã=1466, ADL=1689)  cửa_sổ=236343
== Val ==
  file=949 (ngã=435, ADL=514)  cửa_sổ=71801
== Test ==
  file=971 (ngã=445, ADL=526)  cửa_sổ=68791

GLOBAL_MEAN = [0.02160000056028366, -0.7200999855995178, -0.012900000438094139, -0.7932999730110168, 7.504899978637695, -0.05009999871253967]
GLOBAL_STD  = [0.34529998898506165, 0.5091000199317932, 0.5098999738693237, 36.18899917602539, 43.4015007019043, 21.957199096679688]
Tỉ lệ fall — train 0.0518 | val 0.0507 | test 0.0542

[Firmware] ESP32 PHẢI áp đúng GLOBAL_MEAN/STD và clip ±4 trên dữ liệu thô.


In [4]:
# CELL 2 — Kiến trúc
# =============================================================================
RELU_MAX = 6.0


def _act(x):
    return layers.ReLU(max_value=RELU_MAX)(x)


def identity_block(x, filters=64, reduce_ratio=4, k=3, dilation=1, name=None):
    """Shortcut là dây nối thẳng.

    Mọi block có in = out = 64 và stride = 1, nên conv 1x1 trên shortcut của bản
    gốc đang học một ánh xạ 64->64 lẽ ra là identity. Nó chiếm 59% MAC của block.

    dilation chỉ tác động lên conv2 (lớp duy nhất có kernel 3). Hai conv 1x1
    không có khái niệm giãn nở.
    """
    fr = filters // reduce_ratio
    sc = x
    x = layers.Conv1D(fr, 1, padding='same', name=f"{name}_c1")(x)
    x = layers.BatchNormalization()(x); x = _act(x)
    x = layers.Conv1D(fr, k, padding='same', dilation_rate=dilation, name=f"{name}_c2")(x)
    x = layers.BatchNormalization()(x); x = _act(x)
    x = layers.Conv1D(filters, 1, padding='same', name=f"{name}_c3")(x)
    x = layers.BatchNormalization()(x)
    return _act(layers.Add()([x, sc]))


def build_model(dilations=(1, 1, 1, 1), pool="avg", dropout=0.1,
                filters=64, input_shape=(WIN_SIZE, 6)):
    inp = layers.Input(shape=input_shape, name="imu_input")
    x = layers.Conv1D(filters, 3, padding='same', name="stem_conv")(inp)
    x = layers.BatchNormalization()(x); x = _act(x)
    x = layers.MaxPooling1D(2, strides=2, padding='same', name="stem_pool")(x)

    for i, d in enumerate(dilations):
        x = identity_block(x, filters=filters, dilation=d, name=f"block{i+1}")

    if pool == "avg":
        x = layers.GlobalAveragePooling1D(name="gap")(x)
    else:
        # max = "đỉnh mạnh cỡ nào", avg = "kéo dài bao lâu".
        # Ngã chiếm ~8/25 bước; chỉ lấy trung bình là pha loãng đỉnh xuống 1/3.
        x = layers.Concatenate(name="gap_gmp")([
            layers.GlobalAveragePooling1D()(x),
            layers.GlobalMaxPooling1D()(x)])

    if dropout > 0:
        x = layers.Dropout(dropout, name="drop")(x)
    out = layers.Dense(2, activation=None, name="logits")(x)   # logit thô
    return Model(inp, out)


CONFIGS = {
    "base":        dict(dilations=(1, 1, 1, 1), pool="avg",    augment=False),
    "dilated":     dict(dilations=(1, 1, 2, 4), pool="avgmax", augment=False),
    "dilated_aug": dict(dilations=(1, 1, 2, 4), pool="avgmax", augment=True),
}


def count_macs(model):
    """MAC mỗi lần suy luận. Đây là đại lượng quyết định pin, không phải số KB."""
    tot = 0
    for l in model.layers:
        if isinstance(l, layers.Conv1D):
            tot += int(l.output.shape[1]) * l.filters * l.kernel_size[0] * int(l.input.shape[-1])
        elif isinstance(l, layers.Dense):
            tot += l.units * int(l.input.shape[-1])
    return tot


def receptive_field(dilations):
    """Trường tiếp nhận tính bằng frame gốc. stem k=3 -> 3, maxpool/2 -> 4,
    mỗi block cộng 2*dilation bước ở độ phân giải đã pool = 4*dilation frame gốc."""
    rf = 4
    for d in dilations:
        rf += 4 * d
    return rf


print(f"{'config':<14}{'params':>9}{'MACs':>10}{'RF (ms)':>10}")
for n, c in CONFIGS.items():
    m = build_model(c["dilations"], c["pool"])
    print(f"{n:<14}{m.count_params():>9,}{count_macs(m):>10,}"
          f"{receptive_field(c['dilations'])*10:>10}")
    del m
tf.keras.backend.clear_session()

config           params      MACs   RF (ms)


I0000 00:00:1790483398.085361      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790483398.088155      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


base             14,786   339,328       200
dilated          14,914   339,456       360
dilated_aug      14,914   339,456       360


In [6]:
# CELL 3 — Augmentation: xoay trục IMU (áp trên dữ liệu THÔ)
# =============================================================================
ROT_MAX_DEG = 15.0
SCALE_RANGE = 0.10
NOISE_STD = 0.01

_MEAN_T = tf.constant(GLOBAL_MEAN)
_STD_T = tf.constant(GLOBAL_STD)


def _rand_rotation(b, max_deg):
    """Ma trận xoay ngẫu nhiên nhỏ, mô phỏng cảm biến đeo lệch tư thế."""
    a = tf.random.uniform([b, 3], -max_deg * np.pi / 180, max_deg * np.pi / 180)
    cx, cy, cz = tf.cos(a[:, 0]), tf.cos(a[:, 1]), tf.cos(a[:, 2])
    sx, sy, sz = tf.sin(a[:, 0]), tf.sin(a[:, 1]), tf.sin(a[:, 2])
    o, z = tf.ones_like(cx), tf.zeros_like(cx)
    Rx = tf.reshape(tf.stack([o, z, z, z, cx, -sx, z, sx, cx], 1), [-1, 3, 3])
    Ry = tf.reshape(tf.stack([cy, z, sy, z, o, z, -sy, z, cy], 1), [-1, 3, 3])
    Rz = tf.reshape(tf.stack([cz, -sz, z, sz, cz, z, z, z, o], 1), [-1, 3, 3])
    return tf.matmul(tf.matmul(Rz, Ry), Rx)


def augment(x):
    """x: [B,50,6] THÔ. Cùng một phép xoay cho acc và gyro — đó là vật lý đúng:
    cả hai cùng gắn trên một khối cảm biến bị đeo lệch."""
    b = tf.shape(x)[0]
    R = _rand_rotation(b, ROT_MAX_DEG)
    acc = tf.matmul(x[:, :, :3], R, transpose_b=True)
    gyr = tf.matmul(x[:, :, 3:], R, transpose_b=True)
    x = tf.concat([acc, gyr], -1)
    x = x * tf.random.uniform([b, 1, 6], 1 - SCALE_RANGE, 1 + SCALE_RANGE)
    return x + tf.random.normal(tf.shape(x), 0.0, NOISE_STD)


def norm_tf(x):
    return tf.clip_by_value((x - _MEAN_T) / _STD_T, -CLIP_RANGE, CLIP_RANGE)


def make_train_ds(seed, do_augment):
    ds = tf.data.Dataset.from_tensor_slices((Xtr_raw, ytr_oh))
    ds = ds.shuffle(min(len(Xtr_raw), 100_000), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if do_augment:
        ds = ds.map(lambda x, y: (norm_tf(augment(x)), y), tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda x, y: (norm_tf(x), y), tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

In [7]:
# CELL 4 — Metric, đánh giá, lượng tử hoá
# =============================================================================
PROB_THR = 0.7
MARGIN_THR = float(np.log(PROB_THR / (1 - PROB_THR)))
N_GRID = (1, 2, 3, 4, 5)
P_GRID = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
LEAD_MIN_MS = 200.0       # túi khí cần ~80-120 ms để bung


class PRAUC(tf.keras.metrics.AUC):
    """PR-AUC cho lớp fall.

    val_accuracy vô dụng ở đây: lớp fall chỉ 5.2%, nên accuracy bị lớp đa số nuốt
    và dao động 0.16 điểm suốt 31 epoch — chọn epoch bằng tung đồng xu.
    PR-AUC không đếm 65k cửa sổ âm đoán đúng, chỉ nhìn phần đuôi false positive,
    và độc lập hoàn toàn với p lẫn N nên không rò rỉ điểm vận hành vào việc train.
    """

    def __init__(self, **kw):
        super().__init__(curve='PR', name='prauc', **kw)

    def update_state(self, y_true, y_pred, sample_weight=None):
        return super().update_state(y_true[:, 1], tf.nn.softmax(y_pred)[:, 1], sample_weight)


def predict_margin(model, X, bs=1024):
    z = model.predict(X, batch_size=bs, verbose=0)
    return (z[:, 1] - z[:, 0]).astype(np.float32)


def file_metrics(margins, fids, ftrue, impacts, thr=MARGIN_THR, n=1,
                 win=WIN_SIZE, step=STEP_TRAIN, fs=FS, return_fp=False):
    """Metric mức file theo đúng định nghĩa bài báo.
    Phát hiện = có n cửa sổ LIÊN TIẾP margin >= thr.
    lead = impact - mép cuối cửa sổ hoàn tất chuỗi."""
    df = pd.DataFrame({"f": fids, "p": (margins >= thr).astype(np.int8)})
    TP = FP = TN = FN = 0
    leads, fp_files = [], []
    for f, g in df.groupby("f", sort=False):
        run, det = 0, None
        for i, v in enumerate(g.p.values):
            run = run + 1 if v else 0
            if run >= n:
                det = i
                break
        if ftrue[f] == 1:
            if det is not None:
                TP += 1
                if f in impacts:
                    leads.append(impacts[f] / fs * 1000 - (det * step + win) / fs * 1000)
            else:
                FN += 1
        else:
            if det is not None:
                FP += 1; fp_files.append(f)
            else:
                TN += 1
    sens = TP / (TP + FN) * 100 if TP + FN else np.nan
    spec = TN / (TN + FP) * 100 if TN + FP else np.nan
    r = {"TP": TP, "FP": FP, "TN": TN, "FN": FN, "sens": sens, "spec": spec,
         "acc": (TP + TN) / max(TP + FP + TN + FN, 1) * 100,
         "lead": float(np.mean(leads)) if leads else np.nan,
         "lead_sd": float(np.std(leads)) if leads else np.nan}
    return (r, fp_files) if return_fp else r


class TFLiteRunner:
    """Trả margin ở miền float. zero_point triệt tiêu khi lấy hiệu z1-z0."""

    def __init__(self, content, batch=1024):
        self.it = tf.lite.Interpreter(model_content=content)
        self.batch, self._cur = batch, None
        self._refresh()

    def _refresh(self):
        self.i = self.it.get_input_details()[0]
        self.o = self.it.get_output_details()[0]

    def margins(self, X):
        out = []
        for k in range(0, len(X), self.batch):
            xb = X[k:k + self.batch]
            if self._cur != len(xb):
                self.it.resize_tensor_input(self.i["index"], [len(xb), *X.shape[1:]])
                self.it.allocate_tensors(); self._refresh(); self._cur = len(xb)
            isc, izp = self.i["quantization"]
            osc, _ = self.o["quantization"]
            q = (np.clip(np.round(xb / isc) + izp, -128, 127).astype(np.int8)
                 if self.i["dtype"] == np.int8 else xb.astype(np.float32))
            self.it.set_tensor(self.i["index"], q); self.it.invoke()
            z = self.it.get_tensor(self.o["index"]).astype(np.float32)
            m = z[:, 1] - z[:, 0]
            out.append(m * osc if self.o["dtype"] == np.int8 else m)
        return np.concatenate(out).astype(np.float32)


def quantize(model, seed, tag, n_rep=500):
    rng = np.random.RandomState(seed)
    idx_f, idx_a = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
    rep = np.concatenate([rng.choice(idx_f, n_rep // 2, replace=False),
                          rng.choice(idx_a, n_rep // 2, replace=False)])
    rng.shuffle(rep)

    def rep_ds():
        for i in rep:
            yield [normalize(Xtr_raw[i:i + 1])]

    def conv():
        try:
            return tf.lite.TFLiteConverter.from_keras_model(model)
        except Exception:
            sm = os.path.join(OUT_DIR, f"_sm_{tag}")
            model.export(sm)
            return tf.lite.TFLiteConverter.from_saved_model(sm)

    fp32 = conv().convert()
    c = conv()
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = rep_ds
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = c.inference_output_type = tf.int8
    return fp32, c.convert()

In [9]:
# CELL 5 — Train 1 (config, seed)
# =============================================================================
MAX_EPOCHS = 200
ES_PATIENCE = 30      # PR-AUC nhiễu hơn accuracy ở giai đoạn đầu, 20 dễ dừng oan
LR_PATIENCE = 8


def run_one(cfg_name, seed):
    cfg = CONFIGS[cfg_name]
    tag = f"{cfg_name}_s{seed}"
    print(f"\n{'='*76}\n>>> {tag}  dilations={cfg['dilations']} pool={cfg['pool']} "
          f"augment={cfg['augment']}\n{'='*76}")
    t0 = time.time()

    tf.keras.backend.clear_session()
    np.random.seed(seed); tf.random.set_seed(seed)

    model = build_model(cfg["dilations"], cfg["pool"])
    model.compile(optimizer=optimizers.Adam(1e-3),
                  loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy', PRAUC()])   # KHÔNG class_weight

    cbs = [
        callbacks.EarlyStopping(monitor='val_prauc', mode='max',
                                patience=ES_PATIENCE, restore_best_weights=True, verbose=1),
        # v1: val_loss chạm đáy epoch ~9 rồi tăng đều tới epoch 31 trong khi
        # train accuracy vẫn leo -> lr cố định 1e-3 quá dài ở giai đoạn sau.
        callbacks.ReduceLROnPlateau(monitor='val_prauc', mode='max', factor=0.5,
                                    patience=LR_PATIENCE, min_lr=1e-5, verbose=1),
    ]
    hist = model.fit(make_train_ds(seed, cfg["augment"]), epochs=MAX_EPOCHS,
                     validation_data=(Xva, yva_oh), callbacks=cbs, verbose=2)

    kp = os.path.join(OUT_DIR, f"{tag}.keras")
    model.save(kp)
    fp32, int8 = quantize(model, seed, tag)
    with open(os.path.join(OUT_DIR, f"{tag}_int8.tflite"), "wb") as fh:
        fh.write(int8)

    runner = TFLiteRunner(int8)
    marg = {("float", "val"): predict_margin(model, Xva),
            ("float", "test"): predict_margin(model, Xte),
            ("int8", "val"): runner.margins(Xva),
            ("int8", "test"): runner.margins(Xte)}
    d = marg[("int8", "val")] - marg[("float", "val")]
    corr = float(np.corrcoef(marg[("float", "val")], marg[("int8", "val")])[0, 1])

    meta = {"config": cfg_name, "seed": seed, "params": model.count_params(),
            "macs": count_macs(model), "rf_ms": receptive_field(cfg["dilations"]) * 10,
            "epochs": len(hist.history['loss']),
            "best_prauc": float(max(hist.history['val_prauc'])),
            "keras_KB": os.path.getsize(kp) / 1024,
            "tflite_int8_KB": len(int8) / 1024, "int8_mae": float(np.abs(d).mean()),
            "int8_corr": corr, "secs": time.time() - t0}

    print(f"\n{tag}: {meta['epochs']} epoch, {meta['secs']:.0f}s | "
          f"val PR-AUC {meta['best_prauc']:.4f} | MACs {meta['macs']:,} | "
          f"int8 {meta['tflite_int8_KB']:.1f} KB | corr {corr:.5f}")
    if corr < 0.99:
        print("  [!] corr < 0.99 — tăng n_rep trong quantize()")

    del model, runner
    gc.collect()
    return marg, meta

In [11]:
import json
from pathlib import Path

out = Path(OUT_DIR)
spec_path = out / "dilated_aug_s0_input_spec.json"

spec = {
    "model": "dilated_aug_s0",
    "sample_rate_hz": FS,
    "window_samples": WIN_SIZE,
    "inference_step_samples": STEP_TRAIN,
    "feature_order": FEATURE_COLS,
    "clip_range": CLIP_RANGE,
    "mean": GLOBAL_MEAN.tolist(),
    "std": GLOBAL_STD.tolist(),
    "fall_probability_threshold": PROB_THR,
    "consecutive_windows": 2,
    "train_subjects": list(train_subs),
    "val_subjects": list(val_subs),
    "test_subjects": list(test_subs),
}

spec_path.write_text(
    json.dumps(spec, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Đã lưu:", spec_path)

Đã lưu: /kaggle/working/dilated_aug_s0_input_spec.json


In [12]:
marg, meta = run_one("dilated_aug", 0)


>>> dilated_aug_s0  dilations=(1, 1, 2, 4) pool=avgmax augment=True
Epoch 1/200
3693/3693 - 53s - 14ms/step - accuracy: 0.9702 - loss: 0.1029 - prauc: 0.7163 - val_accuracy: 0.9858 - val_loss: 0.0394 - val_prauc: 0.9383 - learning_rate: 0.0010
Epoch 2/200
3693/3693 - 28s - 8ms/step - accuracy: 0.9841 - loss: 0.0462 - prauc: 0.9107 - val_accuracy: 0.9873 - val_loss: 0.0345 - val_prauc: 0.9436 - learning_rate: 0.0010
Epoch 3/200
3693/3693 - 27s - 7ms/step - accuracy: 0.9858 - loss: 0.0415 - prauc: 0.9250 - val_accuracy: 0.9888 - val_loss: 0.0309 - val_prauc: 0.9528 - learning_rate: 0.0010
Epoch 4/200
3693/3693 - 28s - 7ms/step - accuracy: 0.9868 - loss: 0.0379 - prauc: 0.9352 - val_accuracy: 0.9887 - val_loss: 0.0308 - val_prauc: 0.9529 - learning_rate: 0.0010
Epoch 5/200
3693/3693 - 28s - 8ms/step - accuracy: 0.9874 - loss: 0.0359 - prauc: 0.9411 - val_accuracy: 0.9889 - val_loss: 0.0308 - val_prauc: 0.9561 - learning_rate: 0.0010
Epoch 6/200
3693/3693 - 28s - 7ms/step - accuracy: 0.98

INFO:tensorflow:Assets written to: /tmp/tmpyaq0nkdo/assets


Saved artifact at '/tmp/tmpyaq0nkdo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 6), dtype=tf.float32, name='imu_input')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140301780760528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780761296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780757456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780759184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780761872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780760720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780762256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792035472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792035856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792036816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792036624: Tensor

W0000 00:00:1790485463.220163      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790485463.220189      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1790485463.264585      58 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


INFO:tensorflow:Assets written to: /tmp/tmp8m7qcnil/assets


INFO:tensorflow:Assets written to: /tmp/tmp8m7qcnil/assets


Saved artifact at '/tmp/tmp8m7qcnil'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 6), dtype=tf.float32, name='imu_input')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140301780760528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780761296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780757456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780759184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780761872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780760720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301780762256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792035472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792035856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792036816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140301792036624: Tensor

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1790485466.421288      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790485466.421316      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.



dilated_aug_s0: 65 epoch, 1837s | val PR-AUC 0.9678 | MACs 339,456 | int8 49.9 KB | corr 0.99916


In [13]:
for precision in ("float", "int8"):
    result = file_metrics(
        marg[(precision, "test")],
        idte, fte, impte,
        n=2,
    )
    print(precision, result)

float {'TP': 445, 'FP': 23, 'TN': 503, 'FN': 0, 'sens': 100.0, 'spec': 95.62737642585552, 'acc': 97.6313079299691, 'lead': 264.8988764044944, 'lead_sd': 140.91132321418846}
int8 {'TP': 445, 'FP': 23, 'TN': 503, 'FN': 0, 'sens': 100.0, 'spec': 95.62737642585552, 'acc': 97.6313079299691, 'lead': 261.97752808988764, 'lead_sd': 140.47159555559483}


In [14]:
from pathlib import Path
import tensorflow as tf

model_path = Path("/kaggle/working/dilated_aug_s0_int8.tflite")
spec_path = Path("/kaggle/working/dilated_aug_s0_input_spec.json")

print("Model:", model_path.exists(), model_path.stat().st_size if model_path.exists() else 0)
print("Input spec:", spec_path.exists())

it = tf.lite.Interpreter(model_path=str(model_path))
it.allocate_tensors()

print("Input:", it.get_input_details()[0]["shape"],
      it.get_input_details()[0]["dtype"],
      it.get_input_details()[0]["quantization"])
print("Output:", it.get_output_details()[0]["shape"],
      it.get_output_details()[0]["dtype"],
      it.get_output_details()[0]["quantization"])
print("Ops:", sorted(set(op["op_name"] for op in it._get_ops_details())))

Model: True 51120
Input spec: True
Input: [ 1 50  6] <class 'numpy.int8'> (0.0313725508749485, -1)
Output: [1 2] <class 'numpy.int8'> (0.102272629737854, -8)
Ops: ['ADD', 'BATCH_TO_SPACE_ND', 'CONCATENATION', 'CONV_2D', 'DELEGATE', 'EXPAND_DIMS', 'FULLY_CONNECTED', 'MAX_POOL_2D', 'MEAN', 'MUL', 'REDUCE_MAX', 'RESHAPE', 'SPACE_TO_BATCH_ND']


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [15]:
from pathlib import Path
from zipfile import ZipFile
from IPython.display import FileLink, display

root = Path("/kaggle/working")
zip_path = root / "dilated_aug_s0_esp32.zip"

with ZipFile(zip_path, "w") as z:
    z.write(root / "dilated_aug_s0_int8.tflite",
            arcname="dilated_aug_s0_int8.tflite")
    z.write(root / "dilated_aug_s0_input_spec.json",
            arcname="dilated_aug_s0_input_spec.json")

print("Đã tạo:", zip_path, "—", zip_path.stat().st_size, "bytes")
display(FileLink(str(zip_path)))

Đã tạo: /kaggle/working/dilated_aug_s0_esp32.zip — 52487 bytes


/kaggle/working/dilated_aug_s0_esp32.zip